# 🏋️ NB2 — Baseline : ResNet-18 + Cross-Entropy Standard
**Projet 1 : Data-Centric Deep Learning for Noisy Labels**

Ce notebook couvre :
- Architecture ResNet-18 adaptée CIFAR-10
- Entraînement baseline sur 5 configurations de bruit
- Visualisation des courbes d'apprentissage
- Sauvegarde des résultats pour NB4

⏱️ **Durée estimée : ~2h sur Kaggle T4 GPU**

⚠️ **Prérequis : avoir exécuté NB1 et avoir `shared_config.pkl` disponible**

In [ ]:
import random, os, pickle, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── Charger config partagée ───────────────────────────────────
# Si shared_config.pkl n'est pas trouvé, les valeurs par défaut sont utilisées
try:
    with open('/kaggle/input/nb1-outputs/shared_config.pkl', 'rb') as f:
        cfg = pickle.load(f)
    print('✅ shared_config.pkl chargé depuis /kaggle/input')
except FileNotFoundError:
    try:
        with open('shared_config.pkl', 'rb') as f:
            cfg = pickle.load(f)
        print('✅ shared_config.pkl chargé localement')
    except FileNotFoundError:
        print('⚠️  shared_config.pkl non trouvé — utilisation des valeurs par défaut')
        cfg = {
            'SEED': 42, 'N_CLASSES': 10, 'NOISE_RATES': [0.0, 0.2, 0.4, 0.6],
            'CLASSES': ['airplane','automobile','bird','cat','deer',
                        'dog','frog','horse','ship','truck'],
        }

SEED        = cfg['SEED']
CLASSES     = cfg['CLASSES']
N_CLASSES   = cfg['N_CLASSES']
NOISE_RATES = cfg['NOISE_RATES']

# ── Reproductibilité ──────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.dpi']        = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print(f'💻 Device : {device}')
if torch.cuda.is_available():
    print(f'   GPU : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Recharger CIFAR-10 ────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])
clean_trainset = torchvision.datasets.CIFAR10(
    './data', train=True,  download=True, transform=transform_train)
testset        = torchvision.datasets.CIFAR10(
    './data', train=False, download=True, transform=transform_test)
test_loader    = DataLoader(testset, batch_size=256,
                             shuffle=False, num_workers=2)
print(f'✅ CIFAR-10 rechargé : {len(clean_trainset):,} train | {len(testset):,} test')

---
## 🏗️ Section 1 — Architecture ResNet-18

### Pourquoi ResNet-18 ?

- **Capacité suffisante** : un modèle trop simple ne mémoriserait pas les labels bruités — or c'est précisément ce comportement qu'on veut étudier et contrer.
- **Standard académique** : ResNet-18 est le backbone de référence dans la littérature sur le bruit de labels (Han et al. 2018, Li et al. 2020).
- **Temps d'entraînement raisonnable** : ~15 min/config sur T4 GPU, ce qui permet de tester plusieurs configurations.
- **Adapté à CIFAR-10** : on modifie la première couche et supprime le maxpool car les images 32×32 sont beaucoup plus petites qu'ImageNet (224×224).

In [ ]:
# ── NoisyDataset (copie de NB1 pour autonomie du notebook) ───
class NoisyDataset(Dataset):
    ASYM_MAP = {0:0,1:2,2:0,3:5,4:7,5:3,6:6,7:4,8:8,9:1}

    def __init__(self, dataset, noise_rate=0.0,
                 noise_type='symmetric', seed=42):
        self.dataset      = dataset
        self.noise_rate   = noise_rate
        self.noise_type   = noise_type
        self.clean_labels = np.array(dataset.targets)
        self.noisy_labels = self.clean_labels.copy()

        if noise_rate > 0:
            rng     = np.random.RandomState(seed)
            n_noisy = int(len(self.clean_labels) * noise_rate)
            idx     = rng.choice(len(self.clean_labels), n_noisy, replace=False)
            for i in idx:
                orig = self.clean_labels[i]
                if noise_type == 'symmetric':
                    cands = [c for c in range(10) if c != orig]
                    self.noisy_labels[i] = rng.choice(cands)
                else:
                    t = self.ASYM_MAP[orig]
                    if t != orig: self.noisy_labels[i] = t

        self.noise_mask        = (self.noisy_labels != self.clean_labels)
        self.actual_noise_rate = self.noise_mask.mean()

    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        return img, self.noisy_labels[idx], idx


# ── Modèle ────────────────────────────────────────────────────
def get_resnet18(num_classes=10):
    """
    ResNet-18 adapté pour CIFAR-10 (images 32×32).

    Modifications vs ResNet-18 standard (ImageNet) :
    1. conv1 : kernel 7×7 stride 2  →  3×3 stride 1
       (préserver la résolution spatiale sur des petites images)
    2. maxpool → Identity
       (éviter un downsampling trop agressif dès le début)
    3. fc : 1000 → num_classes
    """
    m = models.resnet18(pretrained=False)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool  = nn.Identity()
    m.fc       = nn.Linear(512, num_classes)
    return m.to(device)


# Test rapide
m_test = get_resnet18()
x_test = torch.randn(4, 3, 32, 32).to(device)
y_test = m_test(x_test)
total  = sum(p.numel() for p in m_test.parameters())
print(f'✅ ResNet-18 CIFAR-10')
print(f'   Input  : {tuple(x_test.shape)}')
print(f'   Output : {tuple(y_test.shape)}')
print(f'   Params : {total:,}  (~{total*4/1e6:.1f} MB float32)')
del m_test, x_test, y_test

---
## 🔁 Section 2 — Boucles d'Entraînement

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), 100. * correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        imgs, lbls = (batch[0], batch[1])
        imgs, lbls = imgs.to(device), lbls.to(device)
        correct += model(imgs).argmax(1).eq(lbls).sum().item()
        total   += lbls.size(0)
    return 100. * correct / total


def run_baseline(noise_rate, noise_type, n_epochs=30, lr=0.1, batch_size=256):
    """
    Entraîne ResNet-18 avec Cross-Entropy standard.
    30 epochs + CosineAnnealingLR : suffisant pour CIFAR-10.
    batch_size=256 : plus rapide que 128 sur GPU sans perte de précision notable.
    """
    torch.manual_seed(SEED); np.random.seed(SEED)

    ds     = NoisyDataset(clean_trainset, noise_rate, noise_type)
    loader = DataLoader(ds, batch_size=batch_size,
                        shuffle=True, num_workers=2, pin_memory=True)

    model     = get_resnet18()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                 momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    for ep in range(n_epochs):
        tr_loss, tr_acc = train_epoch(model, loader, optimizer, criterion)
        te_acc          = evaluate(model, test_loader)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['test_acc'].append(te_acc)
        if (ep + 1) % 10 == 0:
            print(f'  Ep {ep+1:3d}/{n_epochs} | '
                  f'Loss {tr_loss:.3f} | '
                  f'Train {tr_acc:.1f}% | '
                  f'Test {te_acc:.1f}%')

    history['best_test_acc'] = max(history['test_acc'])
    return history

print('✅ Fonctions définies')

---
## 🚀 Section 3 — Experiments Baseline

**5 configurations** choisies pour couvrir :
- Le cas propre (référence absolue)
- Les trois niveaux de bruit symétrique (montée progressive)
- Le bruit asymétrique à 40% (le plus représentatif de Clothing1M)

In [ ]:
# Configurations : (noise_rate, noise_type)
# 5 configs × 30 epochs × ~4 min/epoch ≈ ~2h total
CONFIGS = [
    (0.0, 'symmetric'),    # Référence propre
    (0.2, 'symmetric'),    # Bruit faible
    (0.4, 'symmetric'),    # Bruit modéré — point de rupture
    (0.6, 'symmetric'),    # Bruit élevé
    (0.4, 'asymmetric'),   # Bruit réel (Clothing1M-like)
]

results_baseline = {}
t0 = time.time()

print('🚀 BASELINE — ResNet-18 + Cross-Entropy standard')
print(f'   {len(CONFIGS)} configs × 30 epochs | batch=256 | lr=0.1 cosine')
print('=' * 58)

for noise_rate, noise_type in CONFIGS:
    key  = f'{noise_type}_{noise_rate}'
    t1   = time.time()
    print(f'\n▶ {noise_type:<11} | ε = {noise_rate}')

    hist = run_baseline(noise_rate, noise_type, n_epochs=30)
    results_baseline[key] = hist

    elapsed = time.time() - t1
    print(f'   ✅ Best test acc : {hist["best_test_acc"]:.1f}%  '
          f'({elapsed/60:.1f} min)')

total = time.time() - t0
print(f'\n✅ BASELINE terminé en {total/60:.1f} min')

with open('results_baseline.pkl', 'wb') as f:
    pickle.dump(results_baseline, f)
print('💾 results_baseline.pkl sauvegardé')

---
## 📊 Section 4 — Analyse des Courbes d'Apprentissage

In [ ]:
# ── Courbes train vs test ─────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for ax, (nr, nt) in zip(axes, CONFIGS):
    key    = f'{nt}_{nr}'
    h      = results_baseline[key]
    epochs = range(1, len(h['train_acc']) + 1)

    ax.plot(epochs, h['train_acc'], color='#e74c3c', lw=2,
            linestyle='--', label='Train (bruité)')
    ax.plot(epochs, h['test_acc'],  color='#2980b9', lw=2,
            label='Test (propre)')
    ax.fill_between(epochs,
                    h['test_acc'], h['train_acc'],
                    where=[t > te for t, te in
                           zip(h['train_acc'], h['test_acc'])],
                    alpha=0.12, color='#e74c3c')
    ax.set_title(f'{nt}\nε = {nr}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Epoch')
    ax.set_ylim(15, 105)
    ax.grid(True, alpha=0.3)
    best = h['best_test_acc']
    ax.axhline(best, color='#2980b9', linestyle=':', alpha=0.6)
    ax.text(1, best + 1, f'Best={best:.0f}%', fontsize=8, color='#2980b9')

axes[0].set_ylabel('Accuracy (%)')
axes[0].legend(fontsize=8)

plt.suptitle(
    'Baseline — Courbes d\'apprentissage (Train bruité vs Test propre)\n'
    'La zone rouge = overfitting sur les labels corrompus',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('baseline_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 INTERPRÉTATION DES COURBES :')
print()
print('  ε=0%   : Train ≈ Test → pas d\'overfitting. Référence idéale.')
print()
print('  ε=20%  : Léger gap Train/Test. Le modèle mémorise quelques labels')
print('           bruités mais généralise encore bien. Dégradation modérée.')
print()
print('  ε=40%  : Gap significatif. La train acc monte haut (mémorisation')
print('           du bruit) tandis que la test acc stagne ou chute.')
print('           C\'est le POINT DE RUPTURE de la baseline.')
print()
print('  ε=60%  : EFFONDREMENT. Plus de 60% des labels sont faux — le signal')
print('           d\'apprentissage est dominé par du bruit. La test acc chute')
print('           drastiquement malgré une train acc élevée (mémorisation pure).')
print()
print('  Asymétrique 40% : souvent PLUS difficile que le symétrique 40%.')
print('           Les confusions cat→dog, deer→horse sont cohérentes visuellement,')
print('           le modèle ne peut pas détecter l\'incohérence par la seule loss.')

In [ ]:
# ── Tableau récap baseline ────────────────────────────────────
import pandas as pd

rows = []
for nr, nt in CONFIGS:
    key  = f'{nt}_{nr}'
    h    = results_baseline[key]
    rows.append({
        'Noise Type': nt.capitalize(),
        'Noise Rate': f'{int(nr*100)}%',
        'Best Test Acc': f'{h["best_test_acc"]:.1f}%',
        'Final Test Acc': f'{h["test_acc"][-1]:.1f}%',
        'Max Train Acc': f'{max(h["train_acc"]):.1f}%',
        'Gap Train-Test': f'{max(h["train_acc"]) - h["best_test_acc"]:.1f}pp',
    })

df = pd.DataFrame(rows)
print('\n' + '='*70)
print('TABLEAU RÉCAPITULATIF — BASELINE')
print('='*70)
print(df.to_string(index=False))
print('='*70)
print()
print('📊 Le GAP Train-Test quantifie l\'overfitting au bruit.')
print('   Plus il est grand, plus le modèle mémorise les labels incorrects.')
print()
print('➡️  Lancer NB3_robust_methods.ipynb')